## Problem Statement

To collect restaurant data through web scraping from the EazyDiner platform and perform exploratory data analysis to understand restaurant distribution, pricing trends, customer ratings, and cuisine preferences across cities and regions.

In [1]:
!pip install requests

Defaulting to user installation because normal site-packages is not writeable


In [2]:
!pip install beautifulsoup4

Defaulting to user installation because normal site-packages is not writeable


In [3]:
## import required libraries

import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

In [4]:
def scrape_eazydiner_city(city_name, city_query, pages=10):

    """
    Scrapes restaurant details from the EazyDiner website for a given city.

    Parameters:
    - city_name (str): Name of the city to store in the dataset
    - city_query (str): Query parameter used in the EazyDiner URL
    - pages (int): Number of pages to scrape

    Returns:
    - pandas.DataFrame: DataFrame containing restaurant name, city, location,
      cuisine, cost for two, rating, and restaurant URL
    """
    # Initialize lists to store scraped data
    restaurant_name=[]
    restaurant_url=[]
    city=[]
    cuisine = []
    rating = []
    cost_for_two = []
    location=[]

    # Loop through multiple pages
    for page in range(1, pages + 1):

        # Send request to EazyDiner listing page
        url = f'https://www.eazydiner.com/restaurants?location={city_query}&page={page}'
        
        request = requests.get(url)
        print(f'Page {page} | Status:', request.status_code)  
        html_data = request.text

        # Parse HTML content
        soup = BeautifulSoup(request.text, "html.parser")

        # Extract restaurant cards
        cards = soup.find_all('div', class_='flex align-v-start flex-between padding-t-15 padding-b-15 margin-t-12 relative full-width')
       
        # Stop scraping if no restaurants are found
        if not cards:
            print(f"Stopped at page {page} — no restaurants")
            break
        
        # Extract details from each restaurant card
        for card in cards:
            
            # -------- Restaurant name --------
            name_tag = card.find('h3')
            restaurant_name.append(name_tag.text.strip())
            city.append(city_name)

            # Location and cuisine information
            info_tags = card.find_all("div",class_="relative font-14 grey ellipsis listing_lh_18__xp_N9")

            # ------- Location----------
            if len(info_tags) > 0:
                location.append(info_tags[0].text.strip())
            else:
                location.append(None)

            # -------- Cuisine ----------
            if len(info_tags) > 1:
                cuisine_cost_text = info_tags[1].text
                cuisine_only = cuisine_cost_text.split("·")[0].strip()
                cuisine.append(cuisine_only)
            else:
                cuisine.append(None)

            # -------- Cost for two ----------
            if len(info_tags) > 2:
                cost_text = info_tags[2].text.strip()
                cost_match = re.search(r"₹\s*[\d,]+", cost_text)
                cost_for_two.append(cost_match.group() if cost_match else None)
            else:
                cost_for_two.append(None)
    
            # -------- Restaurant URL --------
            link_tag = name_tag.find("a") if name_tag else None
    
            if link_tag and link_tag.get("href"):
                full_url = "https://www.eazydiner.com" + link_tag["href"]
                restaurant_url.append(full_url)
            else:
                restaurant_url.append(None)
                
            # --------- Rating -----------
            rating_value = None
            svg_tag = card.find("svg")
            if svg_tag:
                text_tag = svg_tag.find("text")
                if text_tag:
                    rating_value = text_tag.text.strip()
            rating.append(rating_value)
            
    # Convert collected data into a DataFrame
    return pd.DataFrame({
        "restaurant_name": restaurant_name,
        "city": city,
        "location": location,
        "cuisine": cuisine,
        "cost_for_two": cost_for_two,
        "rating": rating,
        "restaurant_url": restaurant_url
        })


In [5]:
# Dictionary mapping city names to their corresponding EazyDiner URL query parameters
# Used to scrape restaurant data city-wise across different regions

eazydiner_cities = {
    # -------- North India --------
    "Delhi NCR": "delhi-ncr",
    "Gurgaon": "gurgaon",
    "Noida": "noida",
    "Faridabad": "faridabad",
    "Chandigarh": "chandigarh",
    "Jaipur": "jaipur",

    # -------- South India --------
    "Bangalore": "bengaluru",
    "Chennai": "chennai",
    "Hyderabad": "hyderabad",
    "Kochi": "kochi",
    "Coimbatore": "coimbatore",
    "Trivandrum": "trivandrum",
    "Vizag": "visakhapatnam"
}


In [6]:
# Mapping cities to their respective regions for regional analysis
# Used to create a region column in the dataset

city_region_map = {
    "Bangalore": "South India",
    "Chennai": "South India",
    "Hyderabad": "South India",
    "Kochi": "South India",
    "Coimbatore": "South India",
    "Trivandrum": "South India",
    "Vizag": "South India",

    "Delhi NCR": "North India",
    "Gurgaon": "North India",
    "Noida": "North India",
    "Faridabad": "North India",
    "Chandigarh": "North India",
    "Jaipur": "North India"
}

In [7]:
# Loop through all cities and scrape restaurant data for each city
# Store individual city DataFrames in a list

all_city_dfs = []

for city_name, city_query in eazydiner_cities.items():
    print(f"\nScraping {city_name}...")
    df_city = scrape_eazydiner_city(city_name, city_query, pages=10)
    all_city_dfs.append(df_city)


Scraping Delhi NCR...
Page 1 | Status: 200
Page 2 | Status: 200
Page 3 | Status: 200
Page 4 | Status: 200
Page 5 | Status: 200
Page 6 | Status: 200
Page 7 | Status: 200
Page 8 | Status: 200
Page 9 | Status: 200
Page 10 | Status: 200

Scraping Gurgaon...
Page 1 | Status: 200
Page 2 | Status: 200
Page 3 | Status: 200
Page 4 | Status: 200
Page 5 | Status: 200
Page 6 | Status: 200
Page 7 | Status: 200
Page 8 | Status: 200
Page 9 | Status: 200
Page 10 | Status: 200

Scraping Noida...
Page 1 | Status: 200
Page 2 | Status: 200
Page 3 | Status: 200
Page 4 | Status: 200
Page 5 | Status: 200
Page 6 | Status: 200
Page 7 | Status: 200
Page 8 | Status: 200
Page 9 | Status: 200
Page 10 | Status: 200

Scraping Faridabad...
Page 1 | Status: 200
Page 2 | Status: 200
Page 3 | Status: 200
Page 4 | Status: 200
Page 5 | Status: 200
Page 6 | Status: 200
Page 7 | Status: 200
Page 8 | Status: 200
Page 9 | Status: 200
Page 10 | Status: 200

Scraping Chandigarh...
Page 1 | Status: 200
Page 2 | Status: 200
Page

In [8]:
# Combine all city-wise DataFrames into a single DataFrame

final_df = pd.concat(all_city_dfs, ignore_index=True)

In [9]:
# Map each city to its corresponding region to create a region column

final_df["region"] = final_df["city"].map(city_region_map)

In [10]:
# Preview the combined dataset after scraping all cities

final_df.head()

,restaurant_name,city,location,cuisine,cost_for_two,rating,restaurant_url,region
0,Desi Villagio,Delhi NCR,"Connaught Place (CP), Central Delhi",Indian,₹1300,4.1,https://www.eazydiner.comhttps://www.eazydiner...,North India
1,Cafe Out of the Box Courtyard,Delhi NCR,"Connaught Place (CP), Central Delhi",Multicuisine,₹1200,3.9,https://www.eazydiner.comhttps://www.eazydiner...,North India
2,Hard Rock Cafe,Delhi NCR,"Connaught Place (CP), Central Delhi",Multicuisine,₹1500,4.0,https://www.eazydiner.comhttps://www.eazydiner...,North India
3,Sakura,Delhi NCR,"The Metropolitan Hotel & Spa, New Delhi","Japanese, Sushi",₹4500,4.0,https://www.eazydiner.comhttps://www.eazydiner...,North India
4,Dr. Zombie,Delhi NCR,"Connaught Place (CP), Central Delhi","Italian, Cocktail Menu",₹1000,4.3,https://www.eazydiner.comhttps://www.eazydiner...,North India


In [13]:
# Export the scraped and combined restaurant data to a CSV file for further analysis

final_df.to_csv('eazydiner_restaurants.csv', index=False)